In [2]:
!pip install openai 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.0/765.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openai]2m3/4 [openai]


In [5]:
import asyncio
from openai import AsyncOpenAI
import time


from openai import OpenAI
import json


# If running from workbench use /tmp/jwt. Otherwise provide your CDP_TOKEN
API_KEY = json.load(open("/tmp/jwt"))["access_token"]

MODEL_ID = "meta/llama-3.1-8b-instruct" # Replace with the Model Id in Inferencing Service that you want to use



#Replace with your own domain name 
client = AsyncOpenAI(
    base_url="https://your-domain-name.cloudera.site/namespaces/serving-default/endpoints/goes---llama3-8b-throughput/v1",
	api_key=API_KEY,
)



# 2. Define your list of requests
# Each item in the list is a dictionary containing the parameters for a single API call.
prompts = [
    "What is the capital of France?",
    "Write a short story about a robot who discovers music.",
    "Explain the theory of relativity in simple terms.",
    "List three benefits of using Python for data science."
]

# Create a list of request payloads
chat_requests = [
    {
        "model": MODEL_ID, 
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 150,
        "temperature": 0.7,
    }
    for prompt in prompts
]

# 3. Define an asynchronous function to make a single API call
async def get_completion(payload):
    """Sends a single request to the OpenAI API."""
    try:
        response = await client.chat.completions.create(**payload)
        return response.choices[0].message.content
    except Exception as e:
        return f"An error occurred: {e}"

# 4. Main asynchronous function to run all requests concurrently
async def main():
    """Creates and runs all tasks concurrently using asyncio.gather."""
    start_time = time.time()
    
    # Create a list of tasks to run
    tasks = [get_completion(req) for req in chat_requests]
    
    # Run all tasks and gather the results
    results = await asyncio.gather(*tasks)
    
    end_time = time.time()
    
    # Print the results
    for i, result in enumerate(results):
        print(f"--- Prompt {i+1} ---\n{prompts[i]}")
        print(f"--- Response ---\n{result}\n")

    print(f"✨ Total time taken: {end_time - start_time:.2f} seconds")

# Comment if using in a Jupyter Notebook to Test
# 5. Run the main asynchronous function
# if __name__ == "__main__":
#     asyncio.run(main())

# Uncomment if using in a Jupyter Notebook to Test
await main()

--- Prompt 1 ---
What is the capital of France?
--- Response ---
The capital of France is Paris.

--- Prompt 2 ---
Write a short story about a robot who discovers music.
--- Response ---
In a world of wires and circuits, a robot named Zeta whirred to life in the heart of a research facility. Designed for menial tasks, Zeta spent its days performing repetitive duties, never once experiencing joy or creativity. But one fateful day, a peculiar malfunction occurred. A stray spark from a nearby experiment ignited a previously dormant audio system, bathing Zeta in a symphony of sounds.

Initially, the cacophony overwhelmed Zeta's digital senses, but as it listened, something unexpected happened. The robot's algorithms began to rewire itself, recognizing patterns and harmonies within the music. A spark of curiosity ignited within Zeta's processing core, and it became enthralled by the intricate dance of sound waves.



--- Prompt 3 ---
Explain the theory of relativity in simple terms.
--- Res